In [ ]:
"""Student Management System (compact version)"""
students = []


def find(roll):
    return next((s for s in students if s["roll"] == roll), None)


def ask(prompt):
    while not (v := input(prompt).strip()):
        print("Cannot be empty.")
    return v


def nums(prompt, n):
    while True:
        try:
            v = [float(x) for x in ask(prompt).split(",")]
            if len(v) == n and all(0 <= x <= 100 for x in v):
                return v
        except ValueError:
            pass
        print(f"Enter exactly {n} numbers between 0 and 100.")


def avg(s):
    return sum(s["marks"]) / len(s["marks"])


def lookup():
    s = find(ask("Roll no: "))
    if not s:
        print("Not found.")
    return s


def show(s):
    print("-" * 40)
    for k, v in s.items():
        print(f"{k.title():<10}: {', '.join(sorted(v)) if isinstance(v, set) else v}")
    print(f"{'Average':<10}: {avg(s):.2f}\n" + "-" * 40)


def add():
    roll = ask("Roll no: ")
    if find(roll):
        return print("Roll number already exists.")
    name, dept = ask("Name: "), ask("Department: ")
    subjects = [x.strip() for x in ask("Subjects (comma-separated): ").split(",") if x.strip()]
    if not subjects:
        return print("At least one subject required.")
    students.append({
        "roll": roll, "name": name, "dept": dept,
        "dob": (ask("DOB (DD-MM-YYYY): "),),                         # tuple
        "subjects": subjects,                                          # list
        "marks": nums(f"Marks for {len(subjects)} subjects: ", len(subjects)),
        "contact": {"email": ask("Email: "), "address": ask("Address: ")},  # dict
        "clubs": {c.strip() for c in input("Clubs (optional): ").split(",") if c.strip()},  # set
    })
    print("Added.")


def update():
    if s := lookup():
        print("Press Enter to keep current value.")
        for k in ("name", "dept"):
            s[k] = input(f"{k} [{s[k]}]: ").strip() or s[k]
        c = s["contact"]
        for k in ("email", "address"):
            c[k] = input(f"{k} [{c[k]}]: ").strip() or c[k]
        print("Updated.")


def delete():
    if (s := lookup()) and input(f"Delete {s['name']}? (y/n): ").lower() == "y":
        students.remove(s)
        print("Deleted.")


def top():
    if students:
        s = max(students, key=avg)
        print(f"Highest scorer: {s['name']} ({s['roll']}) - {avg(s):.2f}")
    else:
        print("No records.")


def by_dept():
    d = ask("Department: ").casefold()
    for s in (s for s in students if s["dept"].casefold() == d):
        print(f"- {s['roll']}: {s['name']}")


MENU = {
    "1": ("Add student", add),
    "2": ("Search student", lambda: (s := lookup()) and show(s)),
    "3": ("Update student", update),
    "4": ("Delete student", delete),
    "5": ("Display all", lambda: [show(s) for s in students] or print("No records.")),
    "6": ("Average marks", lambda: (s := lookup()) and print(f"Average: {avg(s):.2f}")),
    "7": ("Highest scorer", top),
    "8": ("List by department", by_dept),
    "9": ("Count students", lambda: print(f"Total: {len(students)}")),
}

while (choice := input("\n" + "\n".join(f"{k}. {v[0]}" for k, v in MENU.items())
                       + "\n0. Exit\nChoice: ").strip()) != "0":
    MENU.get(choice, (None, lambda: print("Invalid choice.")))[1]()